# Japan House Price Prediction - EDA + Location Features

Notebook nay dung code trong `src/` de tranh lap lai logic. Notebook goc `DuDoanGiaNha_2_4_2026.ipynb` van duoc giu nguyen.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.preprocessing.clean_data import load_clean_merged
from src.features.feature_engineering import prepare_model_frame
from src.models.train_xgb import train

## 1. Load, clean va merge toa do

In [ ]:
before_clean, df = load_clean_merged(drop_outliers=True)
df = prepare_model_frame(df)

print('Before cleaning/outlier:', before_clean.shape)
print('After cleaning/location/outlier:', df.shape)
df.head()

## 2. Missing values truoc xu ly

In [ ]:
missing = before_clean.isna().mean().sort_values(ascending=False).head(20) * 100
plt.figure(figsize=(10, 6))
sns.barplot(x=missing.values, y=missing.index)
plt.xlabel('Missing (%)')
plt.title('Top Missing Columns Before Cleaning')
plt.show()

## 3. Target distribution truoc/sau log transform

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(df['price'], bins=50, kde=True, ax=axes[0])
axes[0].set_title('Raw price')
sns.histplot(np.log1p(df['price']), bins=50, kde=True, ax=axes[1])
axes[1].set_title('log1p(price)')
plt.tight_layout()
plt.show()

## 4. Location features

Khong chi dua raw latitude/longitude vao model. Pipeline tao them khoang cach den cac trung tam lon, khoang cach den trung tam gan nhat, log distance, cluster toa do va target encoding theo khu vuc tren train.

In [ ]:
location_cols = [
    'latitude', 'longitude', 'dist_to_tokyo_km', 'dist_to_osaka_km',
    'dist_to_nearest_major_center_km', 'nearest_major_center',
    'Close_to_Tokyo', 'Close_to_greater_Tokyo_area'
]
df[location_cols].head()

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df.sample(min(10000, len(df)), random_state=42), x='longitude', y='latitude', hue='Year', palette='viridis', s=12, alpha=0.45)
plt.title('House Transactions on Japan Coordinates')
plt.show()

## 5. Train XGBoost pipeline

In [ ]:
# Dat JAPAN_HOUSE_MAX_ROWS=0 trong terminal neu muon train full dataset.
metrics = train()
metrics

Sau khi train, xem cac file trong `reports/figures/`: actual vs predicted, residual distribution, feature importance, before/after outlier.